In [1]:
!pip install -q gradio chromadb sentence-transformers pandas plotly huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
import gradio as gr
import pandas as pd
import plotly.express as px
import chromadb
from sentence_transformers import SentenceTransformer
import uuid

# --- INITIALIZE DATABASE & MODELS ---
# Using a local vector store for manufacturing data
client = chromadb.Client()
collection = client.get_or_create_collection(name="mfg_history")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample Data to pre-populate the system
initial_data = [
    {"id": "ISS-101", "problem": "Hydraulic fluid leak near the landing gear strut.", "solution": "Replace the high-pressure seal and check for piston scoring.", "model": "A320", "module": "Landing Gear"},
    {"id": "ISS-102", "problem": "Fastener torque inconsistency in the wing box assembly.", "solution": "Recalibrate pneumatic drivers and implement a secondary torque check.", "model": "B737", "module": "Wings"},
    {"id": "ISS-103", "problem": "Avionics bay cooling fan failure during pre-flight.", "solution": "Clear debris from intake and replace the brushless motor unit.", "model": "A350", "module": "Avionics"}
]

# Add initial data to Vector DB
for item in initial_data:
    embedding = embed_model.encode(item["problem"]).tolist()
    collection.add(
        ids=[item["id"]],
        embeddings=[embedding],
        metadatas=[{"solution": item["solution"], "model": item["model"], "module": item["module"]}],
        documents=[item["problem"]]
    )

# --- AGENT FUNCTIONS ---

def retrieve_and_classify(problem_desc, aircraft_model):
    """Retrieves similar issues and 'classifies' the module."""
    # 1. Search Vector DB
    query_vec = embed_model.encode(problem_desc).tolist()
    results = collection.query(query_embeddings=[query_vec], n_results=1)

    # 2. Extract Data
    if results['ids'][0]:
        match_id = results['ids'][0][0]
        match_sol = results['metadatas'][0][0]['solution']
        match_mod = results['metadatas'][0][0]['module']
        return f"✅ **Matched with {match_id}**\n\n**Suggested Solution:** {match_sol}", match_mod
    return "❌ No similar issues found. Please propose a new solution.", "General"

def generate_lessons_learned(target_model):
    """Simulates agentic summarization of lessons learned."""
    # Filter metadata for the model
    all_data = collection.get(include=['metadatas', 'documents'])
    lessons = []
    for meta, doc in zip(all_data['metadatas'], all_data['documents']):
        if meta['model'] == target_model:
            lessons.append(f"• **Problem:** {doc}\n  **Outcome:** {meta['solution']}")

    if not lessons:
        return "No historical data found for this aircraft model."

    summary_header = f"### Lessons Learned Report: {target_model}\n"
    return summary_header + "\n".join(lessons)

def update_dashboard():
    """Generates analytics for the dashboard tab."""
    all_data = collection.get(include=['metadatas'])
    df = pd.DataFrame([m for m in all_data['metadatas']])
    if df.empty: return None, None

    fig1 = px.pie(df, names='model', title="Issues by Aircraft Model", color_discrete_sequence=px.colors.sequential.RdBu)
    fig2 = px.bar(df, x='module', color='model', title="Problem Distribution by Module")
    return fig1, fig2

# --- GRADIO UI LAYOUT ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ✈️ Aerospace Manufacturing: Agentic AI Intelligence")

    with gr.Tabs():
        # TAB 1: NEW ISSUE ENTRY
        with gr.TabItem("🚀 Issue Analysis & Retrieval"):
            with gr.Row():
                with gr.Column():
                    prob_input = gr.TextArea(label="Problem Description", placeholder="Describe the issue here...")
                    with gr.Row():
                        model_input = gr.Dropdown(["A320", "B737", "A350", "B787"], label="Aircraft Model")
                        module_input = gr.Textbox(label="Module Name (Inferred)", interactive=False)
                    btn_analyze = gr.Button("Analyze Issue", variant="primary")

                with gr.Column():
                    output_solution = gr.Markdown(label="Agent Recommendations")

            btn_analyze.click(
                fn=retrieve_and_classify,
                inputs=[prob_input, model_input],
                outputs=[output_solution, module_input]
            )

        # TAB 2: DASHBOARDS
        with gr.TabItem("📊 Analytics Dashboard"):
            btn_refresh = gr.Button("Refresh Dashboard Data")
            with gr.Row():
                plot1 = gr.Plot()
                plot2 = gr.Plot()

            btn_refresh.click(fn=update_dashboard, outputs=[plot1, plot2])

        # TAB 3: LESSONS LEARNED
        with gr.TabItem("🧠 Lessons Learned"):
            model_select = gr.Dropdown(["A320", "B737", "A350", "B787"], label="Select Model for Summary")
            btn_summary = gr.Button("Generate Model Insights")
            lessons_output = gr.Markdown()

            btn_summary.click(fn=generate_lessons_learned, inputs=model_select, outputs=lessons_output)

# Launch the app
demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_3353/3086032098.py:73: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b70da20fd69b2926f5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
